In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
import os
from langchain_community.document_loaders.pdf import PyMuPDFLoader

C:\Users\mubee\AppData\Local\Temp\ipykernel_12432\180187781.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyMuPDFLoader


## Ingestion Pipeline

In [3]:
def load_all_pdfs():
    folder_path = "./data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path,filename)

            pdf_loader = PyMuPDFLoader(pdf_path)
            doc = pdf_loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print(f"Total loaded pdfs: {num_docs}")
    print(f"Total pages : {len(all_docs)}")

    return all_docs
            

In [4]:
all_documents = load_all_pdfs()

Total loaded pdfs: 2
Total pages : 34


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size=500, chunk_overlap=50):
    text_splitter  = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)

    return chunked_docs

In [8]:
chunks = split_docs(all_documents)


In [7]:
from sentence_transformers import SentenceTransformer
class Embedding_Manager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading Model...")
        self.model = SentenceTransformer(self.model_name)
        print("Model Dimentions: ",self.model.get_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings = self.model.encode(text,show_progress_bar=True)
        print("Embeddings Shape: ",embeddings.shape)
        return embeddings

In [8]:
embedding_manager = Embedding_Manager()

Loading Model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model Dimentions:  384


In [9]:
import chromadb
import uuid

class VectorStoreManager:
    def __init__(self,persist_directory="./vector_store",collection_name="pdf_documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.collection = None
        self.client = None

        self._initialize()
        
    def _initialize(self):
        os.makedirs(self.persist_directory,exist_ok=True)

        self.client = chromadb.PersistentClient(path=self.persist_directory)

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description" :"Vector store collection for pdf embeddings"}
        )

        print("Initialized Vector Store with collections: ",self.collection_name)
        print("Docs in collection: ",self.collection.count())

    def add_document(self,document_chunks,embeddings):

        if(len(document_chunks) != len(embeddings)):
            raise ValueError("num of documents does not match to num of embeddings!")
            
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(document_chunks,embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["Content Length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("Total documents added in Vector Store: ", {len(documents_content)})
        print("Docs in collection: ",self.collection.count())
            

In [10]:
vector_store = VectorStoreManager()

Initialized Vector Store with collections:  pdf_documents
Docs in collection:  799


In [11]:
texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embeddings Shape:  (311, 384)


In [12]:
vector_store.add_document(chunks,embeddings)

Total documents added in Vector Store:  {311}
Docs in collection:  1110


## Retrieval Pipeline

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
class RAG_Retriver:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrive(self,query,top_k=5, score_threshold=0.0):

        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results = top_k
        )

        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i , (doc_id,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):
                similarity_score = 1 - distance

                if similarity_score > score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                        "metadata":metadata,
                        "document":document,
                        "distance":distance,
                        "similarity_score":similarity_score,
                        "rank":i + 1
                    })
        else:
            print("no documents found!")
        
        print(f"Retrived Documents: {len(retrieved_docs)}")
        return retrieved_docs

In [15]:
rag_retriver = RAG_Retriver(embedding_manager,vector_store)

In [16]:
context = rag_retriver.retrive("What is Pure Psychology")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Shape:  (1, 384)
Retrived Documents: 5


In [17]:
context

[{'id': 'doc_f15b99cc-242d-4121-93df-de2ab6235837',
  'metadata': {'keywords': '',
   'moddate': 'D:20240221113308',
   'doc_index': 14,
   'creationDate': 'D:20240221113308',
   'modDate': 'D:20240221113308',
   'file_path': './data\\HUM122_Handouts_Lecture01.pdf',
   'trapped': '',
   'Content Length': 486,
   'source': './data\\HUM122_Handouts_Lecture01.pdf',
   'creator': 'Microsoft® Office Word 2007',
   'creationdate': 'D:20240221113308',
   'producer': 'Microsoft® Office Word 2007',
   'format': 'PDF 1.5',
   'subject': '',
   'author': 'Designer',
   'page': 3,
   'total_pages': 13,
   'title': ''},
  'document': 'BRANCHES OF PSYCHOLOGY \nIt has two main branches: \n1. Pure Psychology: It deals with the psychological research and data which helps to \nformulate the  principles of activity \n2. Applied Psychology: It applies the information given by Pure Psychology, to the problems \nof actual life. \n \n1. Pure Psychology \nPsychology is the scientific study of human and animal

In [ ]:
GROQ_API_KEY=""

In [19]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key = GROQ_API_KEY,
    model="openai/gpt-oss-120b",  
    temperature=0.1,
    max_tokens=1024,
)


In [20]:
def generate_query(query,rag_retriver,llm,top_k=3):
    results = rag_retriver.retrive(query)
    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("We found no context based on query.")
    prompt = f"""You are a helpful assistant.Generate simple text. No Markdown. Use given context to generate the answer. 
                Context : {context}"""
    
    messages = [
        ("system", prompt),
        ("human", query),
    ]
    ai_msg = llm.invoke(messages)

    return ai_msg
    

In [21]:
response = generate_query("What is RAG",rag_retriver,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings Shape:  (1, 384)
Retrived Documents: 5


In [22]:
print(response.content)

RAG stands for Retrieval‑Augmented Generation. It is an approach in natural‑language processing that combines two steps: first, a retrieval component searches a large external knowledge source (such as documents, databases, or the web) to find relevant pieces of information for a given query; second, a generative language model uses both the original query and the retrieved passages to produce a response. By grounding the generation in up‑to‑date factual material, RAG aims to improve the accuracy, relevance, and factual consistency of the output compared with a language model that relies only on its internal parameters.
